In [ ]:
# ==============================================================================
# SETUP
# ==============================================================================
from notebooks.helpers import IncrementalPipeline, TableConfig, safe_count, setup_logger, generate_batch_id
from datetime import date, datetime
import pandas as pd
from notebooks.helpers.silver_transforms import (
    transform_customer_full_pipeline,
    transform_staff_full_pipeline,
    transform_manager_full_pipeline,
    transform_store_full_pipeline,
    transform_car_full_pipeline,
    transform_equipment_dimension
)
from notebooks.helpers import (
    write_gold_table,
    build_staff_hierarchy_bridge,
    build_equipment_bridges,
    generate_date_dimension,
    generate_role_playing_date_dimensions,
)

# Setup logging
logger = setup_logger("incremental_dimensions")

# Initialize pipeline
batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

logger.info("Incremental dimension load initialized")

In [ ]:
# ==============================================================================
# SILVER TRANSFORMATION FUNCTIONS (Imported from shared module)
# ==============================================================================


logger.info("Silver transformation functions imported from shared module")

In [ ]:
# ==============================================================================
# PRE-LOAD DEPENDENCY TABLES (Required for transformations)
# ==============================================================================

# Load dependency tables to bronze (no transformations needed)
dependency_tables = ["address", "city", "country", "staff", "car", "inventory_equipment", "equipment"]

# Some dependency tables don't have a watermark column in the source schema
dependency_watermarks = {
    "address": "last_update",
    "city": None,
    "country": None,
    "staff": "last_update",
    "car": "last_update",
    "inventory_equipment": None,
    "equipment": "last_update",
}

# Business keys for merge (defaults to {table}_id if not specified)
dependency_business_keys = {
    "inventory_equipment": "id",
}

logger.info(f"Pre-loading {len(dependency_tables)} dependency tables to bronze")

for table in dependency_tables:
    try:
        watermark_column = dependency_watermarks.get(table, "last_update")
        force_full = watermark_column is None

        logger.info(f"Loading dependency: {table}")
        df = pipeline.bronze_loader.load_incremental(
            table_name=table,
            watermark_column=watermark_column or "last_update",
            force_full=force_full,
        )
        row_count = df.count()

        # Merge to bronze (create if not exists)
        business_key = dependency_business_keys.get(table, f"{table}_id")
        pipeline.bronze_loader.merge_to_bronze(
            df=df,
            table_name=table,
            business_key=business_key
        )

        # Update watermark (only if column exists in loaded data)
        if watermark_column and watermark_column in df.columns:
            load_type = "FULL" if not pipeline.bronze_loader.watermark_manager.has_watermark(table) else "INCREMENTAL"
            pipeline.bronze_loader.update_watermark(
                table_name=table,
                df=df,
                watermark_column=watermark_column,
                load_type=load_type
            )
        else:
            logger.warning(f"No watermark column for {table}; skipping watermark update")

        logger.info(f"✅ {table}: {row_count:,} rows loaded")
    except Exception as e:
        logger.error(f"❌ Failed to load {table}: {str(e)}")
        raise

logger.info("All dependency tables loaded to bronze")

In [ ]:
# ==============================================================================
# DIMENSION LOAD CONFIGURATION (DRY - Configuration Only)
# ==============================================================================

dimension_configs = [
    TableConfig(
        table_name="customer",
        business_key="customer_id",
        surrogate_key="customer_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_customer",
        silver_transform=transform_customer_full_pipeline,
        dependencies=["address", "city", "country"]
    ),
    TableConfig(
        table_name="staff",
        business_key="staff_id",
        surrogate_key="staff_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_staff",
        silver_transform=transform_staff_full_pipeline,
        dependencies=["address", "city", "country"]
    ),
    TableConfig(
        table_name="staff",
        business_key="staff_id",
        surrogate_key="staff_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_manager",
        silver_transform=transform_manager_full_pipeline,
        dependencies=["address", "city", "country"]
    ),
    TableConfig(
        table_name="store",
        business_key="store_id",
        surrogate_key="store_key",
        watermark_column="last_update",
        scd_type=2,  # SCD Type 2 for manager changes
        tracking_columns=["store_manager_id", "store_manager_first_name", "store_manager_last_name"],
        gold_table_name="dim_store",
        silver_transform=transform_store_full_pipeline,
        dependencies=["staff", "address", "city", "country"]
    ),
    TableConfig(
        table_name="inventory",
        business_key="inventory_id",
        surrogate_key="car_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_car",
        silver_transform=transform_car_full_pipeline,
        dependencies=["car", "inventory_equipment", "equipment"]
    ),
    TableConfig(
        table_name="equipment",
        business_key="equipment_id",
        surrogate_key="equipment_key",
        watermark_column="last_update",
        scd_type=1,
        gold_table_name="dim_equipment",
        silver_transform=transform_equipment_dimension
    ),
]

logger.info(f"Configured {len(dimension_configs)} dimension tables")

In [ ]:
# ==============================================================================
# EXECUTE INCREMENTAL LOAD (DRY - Single Function Call)
# ==============================================================================

# Pre-load snapshot
print("Dimension Table Row Counts (Before):")
print(f"dim_customer: {safe_count(spark, 'dim_customer')}")
print(f"dim_staff: {safe_count(spark, 'dim_staff')}")
print(f"dim_manager: {safe_count(spark, 'dim_manager')}")
print(f"dim_store: {safe_count(spark, 'dim_store')}")
print(f"dim_car: {safe_count(spark, 'dim_car')}")
print(f"dim_equipment: {safe_count(spark, 'dim_equipment')}")

print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

# Load all dimensions incrementally
results = pipeline.load_tables(dimension_configs, force_full=False)

# Display results
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# ==============================================================================
# ROLE-PLAYING DATE DIMENSIONS (Create if missing)
# ==============================================================================

ROLE_PLAYING_DATES = [
    ("dim_service_date", "service_date_key"),
    ("dim_rental_date", "rental_date_key"),
    ("dim_return_date", "return_date_key"),
    ("dim_payment_date", "payment_date_key"),
    ("dim_payment_deadline_date", "payment_deadline_date_key"),
]

if not spark.catalog.tableExists("wheelie.gold.dim_date"):
    logger.info("dim_date missing; generating base date dimension")
    dim_date = generate_date_dimension(
        spark,
        start_date=date(2000, 1, 1),
        end_date=date(2027, 12, 31),
    )
    write_gold_table(dim_date, "dim_date", mode="overwrite")
else:
    dim_date = spark.table("wheelie.gold.dim_date")

missing_role_dims = [
    (table_name, key_name)
    for table_name, key_name in ROLE_PLAYING_DATES
    if not spark.catalog.tableExists(f"wheelie.gold.{table_name}")
]

if missing_role_dims:
    logger.info(f"Creating {len(missing_role_dims)} role-playing date dimensions")
    role_playing_dims = generate_role_playing_date_dimensions(dim_date, missing_role_dims)
    for table_name, df in role_playing_dims:
        write_gold_table(df, table_name, mode="overwrite")
else:
    logger.info("All role-playing date dimensions already exist")


In [ ]:
# ==============================================================================
# REBUILD BRIDGES (Derived Tables)
# ==============================================================================

logger.info("Rebuilding bridge tables")

# Staff hierarchy bridge
staff_bronze = spark.table("wheelie.bronze.staff")
bridge_staff_hierarchy = build_staff_hierarchy_bridge(staff_bronze, max_depth=10)
write_gold_table(bridge_staff_hierarchy, "bridge_staff_hierarchy", mode="overwrite")

# Equipment bridges
inventory_equipment_bronze = spark.table("wheelie.bronze.inventory_equipment")
equipment_bronze = spark.table("wheelie.bronze.equipment")
bridge_equipment_group_equipment, bridge_car_equipment = build_equipment_bridges(
    inventory_equipment_bronze
)
write_gold_table(bridge_equipment_group_equipment, "bridge_equipment_group_equipment", mode="overwrite")
write_gold_table(bridge_car_equipment, "bridge_car_equipment", mode="overwrite")

logger.info("Bridge tables rebuilt")


In [ ]:
# Dimension table row counts
print("Dimension Table Row Counts:")
print(f"dim_customer: {safe_count(spark, 'dim_customer')}")
print(f"dim_staff: {safe_count(spark, 'dim_staff')}")
print(f"dim_manager: {safe_count(spark, 'dim_manager')}")
print(f"dim_store: {safe_count(spark, 'dim_store')}")
print(f"dim_car: {safe_count(spark, 'dim_car')}")
print(f"dim_equipment: {safe_count(spark, 'dim_equipment')}")

# Check latest watermarks
print("\nLatest Watermarks:")
display(spark.table("wheelie.monitoring.watermarks"))